In [1]:
from tqdm.auto import tqdm
import prompt_rag
import os
import sys

os.environ['CUDA_VISIBLE_DEVICES'] = '1,2'

dirs = ["../..", ".."]
for _dir in dirs:
    if _dir not in sys.path:
        sys.path.append(_dir)

import covmis, liar2

from swift.llm import (
    ModelType, get_vllm_engine, get_default_template_type,
    get_template, inference_vllm, VllmGenerationConfig
)
from custom import CustomModelType

# model_type = CustomModelType.mixtral_moe_7b_instruct_awq
model_type = CustomModelType.llama_3_70b_instruct_awq
# model_type = CustomModelType.solar_instruct_10_7b

llm_engine = get_vllm_engine(
    model_type, 
    # torch_dtype=torch.float16,  # 检查正确的数据类型！！！！
    tensor_parallel_size=2,
    max_model_len=4096,
    # gpu_memory_utilization=0.92,
    # model_id_or_path="/home/css/models/Mixtral-8x7B-Instruct-v0.1-GPTQ-int4",
    max_num_seqs=16,
    engine_kwargs = {
        # "enforce_eager": True,
        "seed": 42,
    }
)

template_type = get_default_template_type(model_type)
template = get_template(template_type, llm_engine.hf_tokenizer)

generation_config = VllmGenerationConfig(
    max_tokens=2048,
    temperature=0,
    seed=42
)

get_resp_list = lambda request_list : inference_vllm(
    llm_engine, template, request_list, 
    generation_config=generation_config, 
    use_tqdm=True, 
)


[INFO:swift] Successfully registered `/home/hanlv/workspace/code/research/infodemic/LLM/swift2/swift/llm/data/dataset_info.json`
[INFO:swift] Loading the model using model_dir: /home/css/models/llama-3-70b-instruct-awq
[INFO:swift] model_config: LlamaConfig {
  "_name_or_path": "/home/css/models/llama-3-70b-instruct-awq",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 8192,
  "initializer_range": 0.02,
  "intermediate_size": 28672,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 64,
  "num_hidden_layers": 80,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "quantization_config": {
    "bits": 4,
    "group_size": 128,
    "modules_to_not_convert": null,
    "quant_method": "awq",
    "version": "gemm",
    "zero_point": true
  },
  "rms_norm_eps": 1e-05,
  "

WARNING 01-19 14:53:31 config.py:244] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 01-19 14:53:31 config.py:698] Defaulting to use mp for distributed inference
INFO 01-19 14:53:31 llm_engine.py:169] Initializing an LLM engine (v0.5.1) with config: model='/home/css/models/llama-3-70b-instruct-awq', speculative_config=None, tokenizer='/home/css/models/llama-3-70b-instruct-awq', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=2, pipeline_parallel_size=1, disable_custom_all_reduce=True, quantization=awq, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None), s

In [2]:
K = 5
sort = False

prior_knowledge_version = "1"
search_engine = "brave"
model_name = 'llama3'
dataset = 'covmis' # liar2 covmis
wsc_version = "none"
data_version = '2'


test_brave_serach_dir = '/home/hanlv/workspace/data/machine_learning/dataset/research/misinformation_dataset/COVMIS2/training2/test_brave_search'
search_dates = sorted([i[:10] for i in os.listdir(test_brave_serach_dir)])

for data_type in ['test']:

    with tqdm(total=len(search_dates), desc='Outer Loop', position=0) as pbar_outer:
        for i in range(len(search_dates)):
            search_date = search_dates[i]
            pbar_outer.set_description(f'Outer Loop: Search date({search_date})')

            prompt_rag.update_train_search_llm(
                model_name, get_resp_list, search_engine,
                dataset, prior_knowledge_version,
                data_type=data_type, data_version=data_version, search_date=search_date,
                K=K, sort=sort, # wsc_version=wsc_version
                save2_data_llm=True
            )
            pbar_outer.update(1)


Outer Loop:   0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/1220 [00:00<?, ?it/s]

WARNING 01-19 14:56:01 scheduler.py:1112] Sequence group 71 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=1
